# 01 CounterFact 数据概览：选哪种关系？

**目的**：下载 CounterFact，看看有哪些关系、每种有多少条、长什么样，决定后面实验用哪一种。

**这一步不需要 GPU。** 右侧 Accelerator 选 **None**（不消耗 GPU 额度），Internet 打开。

下载地址和 EasyEdit 官方示例里用的一样。

## 1. 下载 CounterFact

In [ ]:
import json, os, urllib.request
from collections import Counter
import pandas as pd

URL = 'https://memit.baulab.info/data/dsets/counterfact.json'
PATH = '/kaggle/working/counterfact.json'
if not os.path.exists(PATH):
    urllib.request.urlretrieve(URL, PATH)
data = json.load(open(PATH))
print('Number of records:', len(data))

## 2. 看一条完整的数据长什么样

重点看 `requested_rewrite`（要做的编辑）、`paraphrase_prompts`（换说法，测泛化）、`neighborhood_prompts`（同一关系、不同主语、不该被改的事实，测副作用）。

In [ ]:
print(json.dumps(data[0], indent=2, ensure_ascii=False)[:2500])

## 3. 每种关系有多少条？

`relation_id` 是 Wikidata 的关系编号，比如 P103 是"母语"。表里同时显示这种关系最常见的句子模板和一个例子。

In [ ]:
rows = []
for rec in data:
    rr = rec['requested_rewrite']
    rows.append(dict(
        relation=rr.get('relation_id'),
        template=rr['prompt'],
        subject=rr['subject'],
        true=rr['target_true']['str'],
        new=rr['target_new']['str'],
        n_neighborhood=len(rec.get('neighborhood_prompts', [])),
    ))
df = pd.DataFrame(rows)

summary = (df.groupby('relation')
             .agg(count=('subject', 'size'),
                  top_template=('template', lambda s: s.value_counts().index[0]),
                  n_templates=('template', 'nunique'),
                  n_true_answers=('true', 'nunique'),
                  example_true=('true', 'first'),
                  example_new=('new', 'first'))
             .sort_values('count', ascending=False))
pd.set_option('display.max_colwidth', 60)
display(summary)
summary.to_csv('/kaggle/working/counterfact_relations.csv')

**怎么读这张表：**
- `count`：这种关系有多少条编辑。我们希望至少 200 条左右。
- `top_template`：最常见的句子模板，`{}` 是主语的位置。
- `n_templates`：同一关系有几种不同的句子写法。
- `n_true_answers`：原答案有多少种不同的取值。比如"母语"可能只有几十种语言，"首都"可能有几百个城市。**取值越少，不同编辑之间的新答案重复得越多，越可能有共同方向。** 这是选关系时要考虑的一点。

## 4. 找出和"首都"有关的关系

In [ ]:
cap = df[df.template.str.contains('capital', case=False)]
display(cap.groupby('relation').size().rename('count'))
display(cap.head(10))

## 5. 看几个候选关系的样例

把你感兴趣的关系编号填进 `CANDIDATES`，每种看 5 条。

In [ ]:
CANDIDATES = list(summary.index[:4])          # default: the 4 largest relations; change as you like
for r in CANDIDATES:
    print(f'===== {r} ({summary.loc[r, "count"]} records)')
    for _, x in df[df.relation == r].sample(5, random_state=0).iterrows():
        print(f"  {x.template.format(x.subject):55}  true: {x.true:15} new: {x.new}")

## 6. 答案有几个 token？

我们用"平均对数概率"比较原答案和新答案。答案只有 1 个 token 时，这个比较最干净；多 token 的答案会稍微复杂一些。这里只需要 tokenizer，不加载模型。

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('openai-community/gpt2-xl')

df['true_ntok'] = df.true.map(lambda s: len(tok(' ' + s)['input_ids']))
df['new_ntok'] = df.new.map(lambda s: len(tok(' ' + s)['input_ids']))
df['both_single'] = (df.true_ntok == 1) & (df.new_ntok == 1)
display(df.groupby('relation').both_single.agg(['sum', 'mean']).rename(
    columns={'sum': 'n_both_single_token', 'mean': 'fraction'})
    .sort_values('n_both_single_token', ascending=False).round(2))

## 7. 记下你的选择和理由（发给我）

1. 你想选哪种关系？它有多少条？
2. 原答案有多少种取值？新答案呢？
3. 两个答案都是单 token 的有多少条？

选关系时可以考虑三点：条数够多（≥200）、答案取值不要太分散、尽量是单 token。也可以选两种：一种主要研究，另一种用来做"跨关系对照"。